# 🧬 EA-F-001 HANA KIM — Elite Flux LoRA Training Pipeline
**Version**: Production v2.0 | **Base Model**: FLUX.1-dev | **Trainer**: ostris/ai-toolkit

This notebook trains the Hana Kim character identity LoRA optimized for:
- ✅ **Face consistency** across all poses and outfits
- ✅ **Skin texture fidelity** (pores, peach fuzz, translucency)
- ✅ **Identity markers** (beauty mark, asymmetric brows, amber iris)
- ✅ **ComfyUI + IPAdapter + ControlNet** compatible output

**Expected training time**: ~45–90 min on L4 GPU (24GB VRAM)

---
## ⚙️ STEP 1 — Install ai-toolkit & Dependencies

In [ ]:
import os, subprocess, sys

STUDIO_ROOT = "/teamspace/studios/this_studio"
TOOLKIT_DIR = f"{STUDIO_ROOT}/ai-toolkit"
OUTPUT_DIR  = f"{STUDIO_ROOT}/output"
DATASET_DIR = f"{STUDIO_ROOT}/character_id"

# Clone ai-toolkit (ostris) — the gold standard Flux LoRA trainer
if not os.path.exists(TOOLKIT_DIR):
    !git clone --recursive https://github.com/ostris/ai-toolkit.git {TOOLKIT_DIR}
else:
    print('ai-toolkit already cloned, pulling latest...')
    !git -C {TOOLKIT_DIR} pull

%cd {TOOLKIT_DIR}

# Install all dependencies
!pip install -r requirements.txt -q
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install bitsandbytes -q  # Required for AdamW8bit optimizer

# Verify GPU
import torch
print(f'\n🖥️  GPU: {torch.cuda.get_device_name(0)}')
print(f'💾  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'✅  CUDA available: {torch.cuda.is_available()}')

---
## 📁 STEP 2 — Setup Dataset & Verify Captions
Upload `character_id.zip` via the sidebar, then run this cell.

In [ ]:
import os, glob

STUDIO_ROOT = "/teamspace/studios/this_studio"
DATASET_DIR = f"{STUDIO_ROOT}/character_id"
TRAIN_DIR   = f"{DATASET_DIR}/train"

# Unzip if needed
ZIP_PATH = f"{STUDIO_ROOT}/character_id.zip"
if os.path.exists(ZIP_PATH) and not os.path.exists(TRAIN_DIR):
    print("Unzipping dataset...")
    !unzip -q {ZIP_PATH} -d {STUDIO_ROOT}
    print("✅ Unzipped!")
elif os.path.exists(TRAIN_DIR):
    print("✅ Dataset already extracted.")
else:
    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(f"{DATASET_DIR}/identity_anchors", exist_ok=True)
    os.makedirs(f"{DATASET_DIR}/validation", exist_ok=True)
    print("⚠️  Created empty directories. Upload your images to:", TRAIN_DIR)

# Validate dataset
images = glob.glob(f"{TRAIN_DIR}/*.jpg") + glob.glob(f"{TRAIN_DIR}/*.jpeg") + \
         glob.glob(f"{TRAIN_DIR}/*.png") + glob.glob(f"{TRAIN_DIR}/*.webp")
captions = glob.glob(f"{TRAIN_DIR}/*.txt")

print(f'\n📊 Dataset Summary:')
print(f'  🖼️  Images   : {len(images)}')
print(f'  📝 Captions : {len(captions)}')
print(f'  ❌ Missing  : {len(images) - len(captions)} images without captions')

if len(images) < 30:
    print('\n⚠️  WARNING: You have fewer than 30 training images.')
    print('   Minimum 30 recommended. Elite quality needs 60-80 images.')
elif len(images) >= 60:
    print(f'\n🏆 EXCELLENT: {len(images)} images. This is professional-tier training data!')
else:
    print(f'\n✅ GOOD: {len(images)} images. Usable — add more for higher consistency.')

# Check for images missing captions
img_basenames = {os.path.splitext(os.path.basename(f))[0] for f in images}
cap_basenames = {os.path.splitext(os.path.basename(f))[0] for f in captions}
missing = img_basenames - cap_basenames
if missing:
    print(f'\n⚠️  Images without captions (will be skipped in training):')
    for m in missing: print(f'   - {m}')

---
## 🤖 STEP 2B — Auto-Caption Missing Images (Optional but Recommended)

In [ ]:
# Auto-generate captions for any image missing a .txt file
# Uses the HANA master prompt + image-specific descriptors

import os, glob

TRAIN_DIR = "/teamspace/studios/this_studio/character_id/train"
TRIGGER   = "ea_f_001_hana_kim"
BASE_IDENTITY = "Hana Kim, 28-year-old East Asian Korean woman, balanced oval face, deep brown almond eyes, beauty mark below left jawline, dark chestnut mid-back hair, light beige skin, visible pores, natural skin grain"

# Manual caption templates — edit these for your specific images
# Key: filename stem, Value: additional descriptors
MANUAL_CAPTIONS = {
    # Add your image-specific captions here
    # Example:
    # "my_image_01": "full body, wearing ivory silk blouse and tailored camel coat, standing front, indoor studio",
}

images = glob.glob(f"{TRAIN_DIR}/*.jpg") + glob.glob(f"{TRAIN_DIR}/*.jpeg") + \
         glob.glob(f"{TRAIN_DIR}/*.png") + glob.glob(f"{TRAIN_DIR}/*.webp")

generated_count = 0
for img_path in images:
    stem = os.path.splitext(img_path)[0]
    txt_path = stem + '.txt'
    if not os.path.exists(txt_path):
        basename = os.path.basename(stem)
        extra = MANUAL_CAPTIONS.get(basename, "editorial fashion portrait, luxury atmosphere, professional photoshoot")
        caption = f"{TRIGGER}, {BASE_IDENTITY}, {extra}, realistic, photorealistic, high detail"
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(caption)
        print(f'  ✍️  Created caption for: {os.path.basename(img_path)}')
        generated_count += 1

if generated_count == 0:
    print('✅ All images already have captions!')
else:
    print(f'\n✅ Auto-generated {generated_count} captions.')
    print('⚠️  Review these captions and edit them to be more specific for better training quality!')

---
## 🔑 STEP 3 — Hugging Face Authentication (Required for FLUX.1-dev)
FLUX.1-dev requires HuggingFace auth. Get your token from: https://huggingface.co/settings/tokens

In [ ]:
# Set your Hugging Face token to download FLUX.1-dev
# Get token at: https://huggingface.co/settings/tokens
# Also accept the model license at: https://huggingface.co/black-forest-labs/FLUX.1-dev

HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # ← REPLACE WITH YOUR ACTUAL TOKEN

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Login to huggingface
!huggingface-cli login --token {HF_TOKEN}
print('✅ Authenticated with Hugging Face!')

---
## 📝 STEP 4 — Write Elite Training Configuration
This config is fine-tuned for maximum identity consistency with Flux LoRA.

In [ ]:
import os, yaml

STUDIO_ROOT = "/teamspace/studios/this_studio"
TOOLKIT_DIR = f"{STUDIO_ROOT}/ai-toolkit"
OUTPUT_DIR  = f"{STUDIO_ROOT}/output"
DATASET_DIR = f"{STUDIO_ROOT}/character_id"

# ╔══════════════════════════════════════════════════════════════╗
# ║       ELITE TRAINING CONFIG — Optimized for HANA Identity   ║
# ╚══════════════════════════════════════════════════════════════╝
config = {
    'job': 'extension',
    'config': {
        'name': 'ea_f_001_hana_kim_lora',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': OUTPUT_DIR,
            'performance_log_every': 100,
            'device': 'cuda:0',
            'trigger_word': 'ea_f_001_hana_kim',

            # ── Network Architecture ────────────────────────────────
            'network': {
                'type': 'lora',
                'linear': 32,           # Rank 32 = higher capacity than 16, better for complex faces
                'linear_alpha': 16,     # Alpha = 16 with rank 32 gives effective scale of 0.5 (prevents overfit)
            },

            # ── Save Strategy ───────────────────────────────────────
            'save': {
                'dtype': 'float16',
                'save_every': 500,
                'max_step_saves_to_keep': 5,  # Keep 5 checkpoints for rollback
            },

            # ── Dataset Configuration ───────────────────────────────
            'datasets': [{
                'folder_path': f"{DATASET_DIR}/train",
                'caption_ext': 'txt',
                'resolution': [512, 768, 1024],  # Multi-res for better generalization
                'shuffle_tokens': False,  # Keep trigger token first for identity lock
                'cache_latents_to_disk': True,    # Speed optimization
                'tag_dropout': 0.0,               # Do NOT drop tags on face identity LoRA
            }],

            # ── Training Hyperparameters ────────────────────────────
            'train': {
                'batch_size': 1,
                'steps': 3000,              # Increased from 2500 for better convergence
                'gradient_accumulation_steps': 2,  # Effective batch = 2 (stability)
                'train_unet': True,
                'train_text_encoder': False,  # Do NOT train text encoder on face LoRA
                'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch',  # CRITICAL: Flux uses flowmatch
                'optimizer': 'adamw8bit',
                'learning_rate': 4e-4,       # Higher LR with rank 32
                'lr_scheduler': 'cosine',    # Cosine decay for smooth convergence
                'lr_warmup_steps': 200,
                'ema_decay': 0.99,           # Exponential moving avg for smoothing
                'dtype': 'bf16',             # BF16 for CUDA stability
                'max_grad_norm': 1.0,        # Gradient clipping
                'timestep_type': 'weighted', # Focus on perceptually important timesteps
            },

            # ── Model Config ────────────────────────────────────────
            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-dev',
                'is_flux': True,
                'quantize': True,           # 4-bit quant to fit in 24GB VRAM
                'low_vram': True,           # Enable for L4 GPU safety
            },

            # ── Validation Sampling ─────────────────────────────────
            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 250,
                'width': 1024,
                'height': 1024,
                'guidance_scale': 3.5,
                'sample_steps': 28,
                'prompts': [
                    'ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, front-facing, wearing ivory silk blouse, indoor studio, soft Rembrandt lighting, beauty mark below left jawline, 85mm portrait, photorealistic',
                    'ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, 3/4 angle left, wearing tailored camel coat, outdoor Parisian garden, overcast daylight, deep brown almond eyes, 105mm portrait',
                    'ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, full body, wearing elegant emerald green halterneck maxi dress, luxury marble studio, natural daylight, walking pose, editorial',
                    'ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, serious expression, wearing cream sleeveless knit dress, art gallery, direct gaze, visible pores, skin texture, side lighting',
                    'ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, profile right, wearing black asymmetric evening gown, outdoor golden hour, dark chestnut hair flyaways, Hasselblad quality',
                ],
                'neg': 'blurry, low quality, distorted, plastic skin, CGI, illustration, bad anatomy, extra limbs',
                'seed': 420691337,
            }
        }]
    },
    'meta': {
        'name': '[name]',
        'version': '1.0',
    }
}

config_dir = f"{TOOLKIT_DIR}/config"
os.makedirs(config_dir, exist_ok=True)
config_path = os.path.join(config_dir, 'hana_kim_elite_config.yaml')

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'✅ Elite config written to: {config_path}')
print(f'\n📋 Config Summary:')
print(f'  🏋️  Steps        : 3,000')
print(f'  📐 LoRA Rank     : 32 (high capacity for face details)')
print(f'  📉 Learning Rate : 4e-4 with cosine decay')
print(f'  ⚡ Optimizer     : AdamW8bit (memory efficient)')
print(f'  🎯 Trigger Token : ea_f_001_hana_kim')
print(f'  📊 Validation    : every 250 steps (5 sample prompts)')

---
## 🚀 STEP 5 — Launch Training
**Make sure your Lightning AI studio is set to L4 GPU or A10G GPU before running!**

⏱️ Expected time: ~45 min (L4) | ~30 min (A10G) | ~20 min (A100)

In [ ]:
import os, time
STUDIO_ROOT = "/teamspace/studios/this_studio"
TOOLKIT_DIR = f"{STUDIO_ROOT}/ai-toolkit"

os.chdir(TOOLKIT_DIR)
start_time = time.time()
print('🚀 Starting HANA KIM LoRA Training...')
print('   Monitor validation samples in output folder every 250 steps.')
print('   Training log will appear below.\n')

!python run.py config/hana_kim_elite_config.yaml

elapsed = time.time() - start_time
print(f'\n✅ Training complete! Total time: {elapsed/60:.1f} minutes')

---
## 📊 STEP 6 — Evaluate Checkpoints & Select Best Model

In [ ]:
import os, glob
from IPython.display import Image, display, HTML

STUDIO_ROOT = "/teamspace/studios/this_studio"
OUTPUT_DIR  = f"{STUDIO_ROOT}/output/ea_f_001_hana_kim_lora"

# List all saved checkpoints
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/*.safetensors"))
print(f'📦 Found {len(checkpoints)} checkpoint(s):\n')
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / 1024**2
    print(f'  📁 {os.path.basename(ckpt)} ({size_mb:.1f} MB)')

# Display validation samples if they exist
sample_dirs = sorted(glob.glob(f"{OUTPUT_DIR}/samples/*"))
if sample_dirs:
    print(f'\n🖼️  Validation samples from training:')
    latest_samples = sorted(glob.glob(f"{sample_dirs[-1]}/*.jpg") + glob.glob(f"{sample_dirs[-1]}/*.png"))
    print(f'  Showing {len(latest_samples)} final-step samples:')
    for img_path in latest_samples[:5]:
        display(HTML(f'<p style="font-size:12px;color:gray">{os.path.basename(img_path)}</p>'))
        display(Image(img_path, width=512))

# Point to final model
final_model = f"{OUTPUT_DIR}/ea_f_001_hana_kim_lora.safetensors"
if os.path.exists(final_model):
    size_mb = os.path.getsize(final_model) / 1024**2
    print(f'\n🏆 FINAL MODEL: {final_model}')
    print(f'   Size: {size_mb:.1f} MB')
    print(f'   ✅ Ready to download and load in ComfyUI!')

---
## 💾 STEP 7 — Download Model to Local Machine

In [ ]:
# Option A: Direct download via browser (recommended)
# Right-click the .safetensors file in the Lightning AI sidebar → Download

# Option B: Copy to a location for easier access
import os, shutil
STUDIO_ROOT = "/teamspace/studios/this_studio"
OUTPUT_DIR  = f"{STUDIO_ROOT}/output/ea_f_001_hana_kim_lora"
DOWNLOAD_DIR = f"{STUDIO_ROOT}/DOWNLOAD_ME"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Copy final LoRA
final_lora = f"{OUTPUT_DIR}/ea_f_001_hana_kim_lora.safetensors"
if os.path.exists(final_lora):
    shutil.copy(final_lora, f"{DOWNLOAD_DIR}/ea_f_001_hana_kim_lora.safetensors")
    print(f'✅ Copied LoRA to: {DOWNLOAD_DIR}/ea_f_001_hana_kim_lora.safetensors')

# Copy best checkpoint (step 2500 or 3000)
import glob
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/*.safetensors"))
for ckpt in checkpoints:
    dest = f"{DOWNLOAD_DIR}/{os.path.basename(ckpt)}"
    shutil.copy(ckpt, dest)
    print(f'✅ Copied checkpoint: {os.path.basename(ckpt)}')

print(f'\n📥 All files ready in: {DOWNLOAD_DIR}')
print('   Find this folder in the Lightning AI sidebar and right-click → Download.')
print('\n📦 After download, place the .safetensors file into:')
print('   ComfyUI/models/loras/ea_f_001_hana_kim_lora.safetensors')

---
## 🔬 BONUS — Quick Test Inference (Verify LoRA Works)

In [ ]:
# Quick sanity test — run inference with your new LoRA
# Uses diffusers for a fast test without needing ComfyUI

!pip install diffusers transformers accelerate sentencepiece -q

from diffusers import FluxPipeline
import torch

STUDIO_ROOT = "/teamspace/studios/this_studio"
LORA_PATH = f"{STUDIO_ROOT}/output/ea_f_001_hana_kim_lora/ea_f_001_hana_kim_lora.safetensors"

if not os.path.exists(LORA_PATH):
    print('⚠️  LoRA not found. Run training first.')
else:
    print('🔄 Loading FLUX.1-dev pipeline...')
    pipe = FluxPipeline.from_pretrained(
        'black-forest-labs/FLUX.1-dev',
        torch_dtype=torch.bfloat16
    ).to('cuda')

    print('🔄 Loading Hana Kim LoRA...')
    pipe.load_lora_weights(LORA_PATH)

    test_prompt = (
        "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, "
        "balanced oval face, beauty mark below left jawline, deep brown almond eyes, "
        "wearing ivory silk blouse, indoor studio, soft lighting, 85mm portrait, photorealistic"
    )

    print('⚡ Generating test image...')
    image = pipe(
        prompt=test_prompt,
        height=1024, width=1024,
        num_inference_steps=28,
        guidance_scale=3.5,
        generator=torch.Generator('cuda').manual_seed(420691337)
    ).images[0]

    output_path = f"{STUDIO_ROOT}/DOWNLOAD_ME/test_inference.png"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    image.save(output_path)

    from IPython.display import Image, display
    print('✅ Test generation complete!')
    display(Image(output_path, width=512))